## RFM Customer Segmentation Analysis 

### 1) Problem Statement:  

This project analyzes customer behaviour for a UK-based online gift retailer using RFM (Recency, Frequency, Monetary) segmentation. Despite having over 540,000 transactions across more than 4,300 customers, the business lacked a clear understanding of which customers were the most valuable, which customers were becoming inactive, and which had likely churned. The analysis groups customers based on their purchasing behaviour by looking at how recently they purchased, how often they buy, and how much they spend. It also helps identify how revenue is distributed across different customer segments. These insights can support more targeted retention campaigns, upselling opportunities for loyal customers, and win-back strategies for inactive customers to improve customer retention and overall revenue.

### 2) Import Libraries and Data:  

This section imports the libraries used throughout the analysis. Pandas is used for exploration, analysis and manipulation . Matplotlib and Seaborn are used for data visualization. The dataset is loaded from a local file path.

In [66]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np 

In [67]:
df = pd.read_csv(r"C:\Users\akhil\Desktop\hustle\Excel Projects\RFM analysis project\data\raw data.csv")
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12-01-2010 08:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12-01-2010 08:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12-01-2010 08:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12-01-2010 08:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12-01-2010 08:26,3.39,17850.0,United Kingdom


### 3) Exploratory Data Analysis:

The exploratory analysis was conducted to understand the structure of the raw dataset and identify data quality issues prior to cleaning. Several key issues were identified, including 135,080 rows with missing CustomerIDs corresponding to anonymous transactions that could not be linked to individual customer profiles, negative quantities representing product returns, cancelled orders marked by InvoiceNo values beginning with “C”, and products with unit prices below £0.10 that appeared to be internal adjustments. These observations formed the basis for the data cleaning decisions described in Section 4.

In [68]:
df.shape

(541909, 8)

In [69]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


In [70]:
df.describe()

,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


In [71]:
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [72]:
df[df['InvoiceNo'].astype(str).str.startswith('C')].shape

(9288, 8)

In [73]:
df[df['UnitPrice'] <= 0]['Description'].value_counts().head(10)

Description
check                     159
?                          47
damages                    45
damaged                    43
found                      25
sold as set on dotcom      20
adjustment                 16
Damaged                    14
thrown away                 9
Unsaleable, destroyed.      9
Name: count, dtype: int64

In [74]:
df[df['UnitPrice'] < 0.10][['Description', 'Quantity','UnitPrice', 'StockCode']].value_counts()

Description                    Quantity  UnitPrice  StockCode
POPART WOODEN PENCILS ASST      100      0.04       16045        59
HOUSE SHAPE PENCIL SHARPENER    48       0.06       16219        25
CARTOON  PENCIL SHARPENERS      80       0.06       16218        24
PIECE OF CAMO STATIONERY SET    108      0.08       16259        15
LETTER SHAPE PENCIL SHARPENER   80       0.06       16216        13
                                                                 ..
wrongly marked carton 22804    -256      0.00       85123A        1
wrongly marked. 23343 in box   -3100     0.00       20713         1
wrongly sold (22719) barcode    170      0.00       22467         1
wrongly sold as sets           -600      0.00       85172         1
?                              -400      0.00       21621         1
Name: count, Length: 1003, dtype: int64

### 4) Data Cleaning

The following cleaning steps were applied to the raw dataset:

1. Removed rows with null CustomerID (135,080 rows) as anonymous 
   transactions cannot be assigned to a customer profile.
2. Removed rows with negative Quantity as these indicate returns 
   which do not represent completed purchases.
3. Removed cancelled orders (InvoiceNo starting with C) as 
   cancellations are not completed purchases.
4. Removed rows with UnitPrice of 0 or below as these indicate 
   internal adjustments and company activity.
5. Removed rows with UnitPrice below £0.10 (15 rows) as these 
   were identified as internal transactions including bank charges 
   and manual adjustments.
6. Converted InvoiceDate to datetime format to enable date 
   based calculations.
7. Converted CustomerID to string as it is an identifier 
   not a numeric value.

Final cleaned dataset: 397,638 rows, 4,338 unique customers, 
0 null values.

In [75]:
df = df[df['CustomerID'].notna()]

In [76]:
df = df[df['Quantity'] > 0]

In [77]:
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]


In [78]:
df = df[df['UnitPrice'] > 0]

In [79]:
df = df[df['UnitPrice'] >= 0.10]


In [80]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'],format='mixed',dayfirst=True)

In [81]:
df['CustomerID'] = df['CustomerID'].astype(str)

### 5) Cleaned Data Verification


Following the cleaning process, the final dataset contains 397638 rows and 8 columns with 0 null values remaining. The minimum order quantity is 1 and the minimum unit price is £0.10, confirming all invalid transactions have been removed. The dataset spans 2010-01-12 to 2011-12-10 and covers 4338 unique customers across 18529 unique invoices.

In [82]:
print(f"Final dataset shape: {df.shape}")
print(f"Null values remaining: {df.isnull().sum().sum()}")
print(f"Min quantity: {df['Quantity'].min()}")
print(f"Min unit price: {df['UnitPrice'].min()}")
print(f"Date range: {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")
print(f"Unique customers: {df['CustomerID'].nunique()}")
print(f"Unique invoices: {df['InvoiceNo'].nunique()}")


Final dataset shape: (397638, 8)
Null values remaining: 0
Min quantity: 1
Min unit price: 0.1
Date range: 2010-01-12 08:26:00 to 2011-12-10 17:19:00
Unique customers: 4338
Unique invoices: 18529


### 6) RFM Calculation 

This section prepares the data for RFM scoring. A TotalPrice column was calculated as the product of Quantity and UnitPrice to derive revenue per transaction. A reference date of 2011-12-11 was set as one day after the last transaction in the dataset, ensuring all recency values are calculated from a consistent fixed point. An RFM dataframe was then created by grouping transactions by CustomerID and aggregating three metrics: the most recent InvoiceDate (LastPurchaseDate), the count of unique invoices (Frequency), and the sum of TotalPrice (Monetary). Finally the Recency column was calculated as the number of days between the reference date and each customer's LastPurchaseDate. 

In [83]:
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
df[['CustomerID', 'InvoiceNo', 'Quantity', 'UnitPrice', 'TotalPrice']].head()

,CustomerID,InvoiceNo,Quantity,UnitPrice,TotalPrice
0,17850.0,536365,6,2.55,15.30
1,17850.0,536365,6,3.39,20.34
2,17850.0,536365,8,2.75,22.00
3,17850.0,536365,6,3.39,20.34
4,17850.0,536365,6,3.39,20.34


In [84]:
reference_date= df['InvoiceDate'].max() + pd.Timedelta(days=1) 

In [85]:
rfm = df.groupby('CustomerID').agg({ 'InvoiceDate': 'max', 'InvoiceNo': 'nunique', 'TotalPrice': 'sum' })
rfm.reset_index(inplace=True)

In [86]:
rfm.columns = ['CustomerID','LastPurchaseDate', 'Frequency', 'Monetary']

In [87]:
rfm['Recency'] = (reference_date- rfm['LastPurchaseDate']).dt.days 

### 7) Outlier Investigtion and Removal 

This section focuses on investigating the distribution and presence of outliers within the Recency, Frequency, and Monetary variables. Descriptive statistics and percentile analysis were performed to better understand the spread and skewness of the data.

The analysis revealed that the 99th percentile of the Monetary variable was approximately 19,880, indicating the presence of a small group of customers with exceptionally high purchase values. Further investigation showed that these customers were primarily wholesale buyers whose purchasing behavior differed significantly from typical retail customers. Since the objective of this analysis is to understand standard customer purchasing patterns, these wholesale customers were excluded from the dataset. In total, 44 customers were removed from the analysis.

In [88]:
rfm['Frequency'].describe()

count    4338.000000
mean        4.271323
std         7.698127
min         1.000000
25%         1.000000
50%         2.000000
75%         5.000000
max       209.000000
Name: Frequency, dtype: float64

In [89]:
rfm['Monetary'].describe()

count      4338.000000
mean       2053.866314
std        8988.535992
min           3.750000
25%         307.037500
50%         674.485000
75%        1661.740000
max      280206.020000
Name: Monetary, dtype: float64

In [90]:
rfm['Recency'].describe()

count    4338.000000
mean      106.470954
std       115.082161
min         1.000000
25%        23.000000
50%        62.000000
75%       162.750000
max       698.000000
Name: Recency, dtype: float64

In [91]:
rfm[rfm['Monetary'] > 50000][['CustomerID', 'Frequency', 'Monetary', 'Recency']]

,CustomerID,Frequency,Monetary,Recency
0,12346.0,1,77183.60,327
55,12415.0,21,124914.53,26
562,13089.0,97,58825.83,7
996,13694.0,50,64815.62,27
1284,14088.0,13,50491.81,12
1289,14096.0,17,65164.79,13
1333,14156.0,55,117375.63,1
1434,14298.0,44,51430.30,54
1689,14646.0,73,280206.02,3
1879,14911.0,201,143787.14,1


In [92]:
print(rfm['Monetary'].quantile(0.99))
print(rfm['Frequency'].quantile(0.99))

19880.99570000001
30.0


In [93]:
wholesale = rfm[rfm['Monetary'] > 19880]['CustomerID'].tolist()
df[df['CustomerID'].isin(wholesale)]['Country'].value_counts()

Country
United Kingdom    33773
EIRE               7065
Netherlands        2076
Australia           714
Singapore           222
Sweden              198
Japan               197
Name: count, dtype: int64

In [94]:
rfm = rfm[rfm['Monetary'] <= 19880]
print(rfm.shape)

(4294, 5)


### 8) RFM Scoring

This section focuses on assigning scores to customers based on their Recency, Frequency, and Monetary performance. To achieve this, the data was divided into bins using the `pd.cut()` function, where each bin was assigned a corresponding score.
A manual binning approach using `pd.cut()` was chosen instead of automatic equal-frequency binning with `pd.qcut()`. This decision was made because the Recency and Frequency variables contained a large number of duplicate values. Using `pd.qcut()` in such cases can lead to overlapping or identical bin edges, making the segmentation unstable and difficult to interpret. Manual binning provided greater control over the score ranges and ensured consistent customer segmentation.  
A key observation from the Frequency analysis was that 1,492 customers, representing approximately 34% of the customer base, had made only a single purchase. This indicates that a significant portion of customers did not return after their initial transaction, highlighting potential issues related to customer retention and repeat engagement.



In [95]:
print(rfm['Monetary'].describe())
print(f"Duplicate Recency values: {rfm['Recency'].duplicated().sum()}")
print(f"Duplicate Monetary values: {rfm['Monetary'].duplicated().sum()}")

count     4294.000000
mean      1410.936064
std       2108.760513
min          3.750000
25%        305.652500
50%        663.730000
75%       1612.415000
max      19824.050000
Name: Monetary, dtype: float64
Duplicate Recency values: 3976
Duplicate Monetary values: 85


In [96]:
rfm['R_score'] = pd.cut(rfm['Recency'],bins=[0, 19, 44, 89, 190, 698], labels=[5, 4, 3, 2, 1])

rfm['F_score'] = pd.cut(rfm['Frequency'],bins=[0, 1, 2, 3, 5, 93],labels=[1, 2, 3, 4, 5])

rfm['M_score'] = pd.cut(rfm['Monetary'],bins=[0, 248.10, 483.65, 920.1, 1998.00, 19824.05],labels=[1, 2, 3, 4, 5])

In [97]:
print("R_score distribution:")
print(rfm['R_score'].value_counts().sort_index())

print("\nF_score distribution:")
print(rfm['F_score'].value_counts().sort_index())

print("\nM_score distribution:")
print(rfm['M_score'].value_counts().sort_index())

R_score distribution:
R_score
5    895
4    825
3    864
2    856
1    854
Name: count, dtype: int64

F_score distribution:
F_score
1    1492
2     836
3     505
4     629
5     832
Name: count, dtype: int64

M_score distribution:
M_score
1    859
2    858
3    859
4    859
5    859
Name: count, dtype: int64


In [98]:
print("Recency ranges:")
print(rfm.groupby('R_score')['Recency'].agg(['min', 'max']))

print("\nFrequency ranges:")
print(rfm.groupby('F_score')['Frequency'].agg(['min', 'max']))

print("\nMonetary ranges:")
print(rfm.groupby('M_score')['Monetary'].agg(['min', 'max']))


Recency ranges:
         min  max
R_score          
5          1   19
4         20   44
3         45   89
2         90  190
1        191  698

Frequency ranges:
         min  max
F_score          
1          1    1
2          2    2
3          3    3
4          4    5
5          6   93

Monetary ranges:
             min       max
M_score                   
1           3.75    248.10
2         248.42    483.40
3         483.65    920.10
4         920.51   1998.00
5        2000.86  19824.05


C:\Users\akhil\AppData\Local\Temp\ipykernel_29104\337660372.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(rfm.groupby('R_score')['Recency'].agg(['min', 'max']))
C:\Users\akhil\AppData\Local\Temp\ipykernel_29104\337660372.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(rfm.groupby('F_score')['Frequency'].agg(['min', 'max']))
C:\Users\akhil\AppData\Local\Temp\ipykernel_29104\337660372.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future 

### 9. RFM Score Concatenation

After assigning the individual R_score, F_score, and M_score values, the three scores were concatenated to generate a combined RFM score for each customer. This consolidated score simplifies customer segmentation and makes it easier to analyse customer behaviour patterns across different groups.

In [99]:
rfm['RFM_Score'] = (rfm['R_score'].astype(str) + 
                     rfm['F_score'].astype(str) + 
                     rfm['M_score'].astype(str))

rfm[['CustomerID', 'Recency', 'Frequency', 'Monetary', 
     'R_score', 'F_score', 'M_score', 'RFM_Score']].head(10)

,CustomerID,Recency,Frequency,Monetary,R_score,F_score,M_score,RFM_Score
1,12347.0,41,7,4310.00,4,5,5,455
2,12348.0,77,4,1797.24,3,4,4,344
3,12349.0,20,1,1757.55,4,1,4,414
4,12350.0,312,1,334.40,1,1,2,112
5,12352.0,74,8,2506.04,3,5,5,355
6,12353.0,205,1,89.00,1,1,1,111
7,12354.0,234,1,1079.40,1,1,4,114
8,12355.0,97,1,459.40,2,1,2,212
9,12356.0,24,3,2811.43,4,3,5,435
10,12357.0,183,1,6207.67,2,1,5,215


### 10) RFM segmentation 

This section segments customers based on their RFM characteristics to identify distinct behavioural groups. In addition to traditional RFM segments such as Champions, Loyal Customers, and At Risk Customers, three dedicated one-time customer segments were introduced. Analysis showed that approximately 35% of customers had made only a single purchase, making them too significant to be grouped into a generic "Other" category.

One-time customers were divided into New Customers (first purchase within 20 days), Recent One-Time Customers (single purchase within the last 21–60 days), and Lapsed One-Time Customers (single purchase more than 60 days ago). This refinement provides greater visibility into customer retention behaviour and reduced the proportion of unclassified customers in the "Other" category from approximately 11.5% to 1.3%.

In [100]:
rfm['R_score'] = rfm['R_score'].astype(int)
rfm['F_score'] = rfm['F_score'].astype(int)
rfm['M_score'] = rfm['M_score'].astype(int)

In [101]:
conditions = [ 
    
# New Customers
(rfm['Frequency'] == 1) &
(rfm['Recency'] <= 20),

# Recent One-Time Customers
(rfm['Frequency'] == 1) &
(rfm['Recency'] > 20) &
(rfm['Recency'] <= 60),

# Lapsed One-Time Customers
(rfm['Frequency'] == 1) &
(rfm['Recency'] > 60),

# Champions
(rfm.R_score >= 4) &
(rfm.F_score >= 4) &
(rfm.M_score >= 4),

# Loyal Customers
(rfm.F_score >= 4) &
(rfm.R_score >= 3),

# Potential Loyalists
(rfm.R_score >= 4) &
(rfm.F_score >= 2) &
(rfm.F_score <= 3),

# Big Ticket Buyers
(rfm.M_score == 5) &
(rfm.F_score <= 2),

# At Risk
(rfm.R_score <= 2) &
(rfm.F_score >= 3) &
(rfm.M_score >= 3),

# Hibernating
(rfm.R_score <= 2) &
(rfm.F_score <= 2),

# Promising
(rfm.R_score >= 3) &
(rfm.F_score == 2),

# Need Nurturing
(rfm.R_score == 3) &
(rfm.F_score == 3),

]
    

In [102]:
choices = [
    'New Customers',
    'Recent One-Time Customers',
    'Lapsed One-Time Customers',
    'Champions',
    'Loyal Customers',
    'Potential Loyalists',
    'Big Ticket Buyers',
    'At Risk',
    'Hibernating',
    'Promising',
    'Need Nurturing'
]

In [103]:
rfm['Segment'] = np.select(
    conditions,
    choices,
    default='Other'
)

In [104]:
rfm['Segment'].value_counts()


Segment
Lapsed One-Time Customers    1125
Champions                     873
Potential Loyalists           486
Loyal Customers               388
Hibernating                   372
At Risk                       315
Recent One-Time Customers     264
Promising                     166
Need Nurturing                129
New Customers                 103
Other                          54
Big Ticket Buyers              19
Name: count, dtype: int64

In [105]:
segment_pct = (
    rfm['Segment']
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

segment_pct

Segment
Lapsed One-Time Customers    26.20
Champions                    20.33
Potential Loyalists          11.32
Loyal Customers               9.04
Hibernating                   8.66
At Risk                       7.34
Recent One-Time Customers     6.15
Promising                     3.87
Need Nurturing                3.00
New Customers                 2.40
Other                         1.26
Big Ticket Buyers             0.44
Name: proportion, dtype: float64

In [106]:
segment_summary = rfm.groupby('Segment').agg({
    'CustomerID': 'count',
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': 'mean'
})

In [107]:
segment_summary.rename(
    columns={'CustomerID': 'CustomerCount'},
    inplace=True
)

In [108]:
segment_summary['CustomerPct'] = (
    segment_summary['CustomerCount']
    / rfm.shape[0]
    * 100
)

In [109]:
segment_summary = segment_summary.round(2)

In [117]:
segment_summary = segment_summary.sort_values(
    by='CustomerCount',
    ascending=False
)
segment_summary

,CustomerCount,Recency,Frequency,Monetary,CustomerPct
Segment,,,,,
Lapsed One-Time Customers,1125,225.38,1.00,364.74,26.20
Champions,873,17.49,9.81,3711.73,20.33
Potential Loyalists,486,21.54,2.43,908.32,11.32
Loyal Customers,388,50.31,5.82,1897.94,9.04
Hibernating,372,192.91,2.00,581.46,8.66
At Risk,315,148.49,4.55,1773.45,7.34
Recent One-Time Customers,264,40.57,1.00,352.95,6.15
Promising,166,65.56,2.00,649.94,3.87
Need Nurturing,129,64.42,3.00,1111.80,3.00


* Lapsed One-Time Customers represent the largest segment (26.2%), indicating a significant customer retention challenge.
* Champions account for 20.3% of customers and generate the highest average revenue and purchase frequency.
* Loyal Customers and Potential Loyalists together represent over 20% of the customer base, providing opportunities for targeted loyalty and retention initiatives.
* The introduction of dedicated One-Time Customer segments reduced the "Other" category from 11.5% to 1.3%, resulting in a more comprehensive customer classification framework.
* At Risk and Lost Customers collectively account for approximately 16% of customers and represent a key audience for reactivation campaigns.


### 11) Business Recommendations

Based on the customer segmentation analysis, targeted marketing actions can be designed for different customer groups. The following analysis identifies the segment with the highest potential return on marketing investment and recommends appropriate business actions.

#### Recommendation 1: Target Potential Loyalists to Increase Conversion into Loyal Customers

In [111]:
segment_summary.loc[
    [
        'Potential Loyalists',
        'Loyal Customers',
        'At Risk',
        'Lapsed One-Time Customers'
    ]
]

,CustomerCount,Recency,Frequency,Monetary,CustomerPct
Segment,,,,,
Potential Loyalists,486,21.54,2.43,908.32,11.32
Loyal Customers,388,50.31,5.82,1897.94,9.04
At Risk,315,148.49,4.55,1773.45,7.34
Lapsed One-Time Customers,1125,225.38,1.00,364.74,26.20


Potential Loyalists are recommended as the primary target segment for future marketing campaigns. This segment has the lowest average recency (21.54 days) among the candidate segments, indicating recent engagement and a higher likelihood of responding to retention efforts. The segment also represents 11.3% of the customer base (486 customers), making it large enough for marketing actions to have a meaningful business impact. Furthermore, customers in this segment have already demonstrated repeat purchase behaviour with an average frequency of 2.43 purchases, suggesting a greater probability of conversion into Loyal Customers compared to one-time buyers. Finally, the segment generates an average monetary value of £908 per customer, and there is significant revenue upside as Loyal Customers spend approximately £1,898 on average.

#### Recommendation 2: Retain Champions 

In [112]:
segment_summary.loc[['Champions']]

,CustomerCount,Recency,Frequency,Monetary,CustomerPct
Segment,,,,,
Champions,873,17.49,9.81,3711.73,20.33


The Champions segment should be a key focus for retention efforts. This segment represents 20.3% of the customer base, making it one of the largest customer groups. More importantly, Champions have the highest average monetary value (£3,711.73), indicating that they contribute significantly to overall business revenue. Their high purchase frequency and low average recency demonstrate strong engagement and loyalty towards the business. Retaining these customers is critical, as the loss of even a small proportion of Champions could have a substantial impact on revenue. Therefore, loyalty rewards, exclusive offers, and personalised marketing campaigns should be used to maintain engagement and encourage continued spending.

#### Recommendation 3: Win Back at Risk Customers

In [113]:
segment_summary.loc[
    [
        'At Risk',
        'Potential Loyalists',
        'Champions'
    ]
]

,CustomerCount,Recency,Frequency,Monetary,CustomerPct
Segment,,,,,
At Risk,315,148.49,4.55,1773.45,7.34
Potential Loyalists,486,21.54,2.43,908.32,11.32
Champions,873,17.49,9.81,3711.73,20.33


The At Risk segment represents customers who were previously valuable to the business but have not made a purchase in a considerable period of time. These customers have a relatively high average frequency (4.55) and monetary value (£1,773), indicating that they were once regular and high-spending customers. However, their average recency of 148 days suggests that they have become disengaged and may be at risk of being lost entirely. Re-engaging this segment presents an opportunity to recover valuable customers and generate additional revenue. Since these customers have already demonstrated repeat purchase behaviour, the likelihood of winning them back is higher than acquiring entirely new customers. Targeted win-back campaigns, loyalty incentives, and personalised promotions should be implemented to encourage repeat purchases and restore engagement.

### 12) Conclusion

The analysis identified distinct customer groups using RFM segmentation and highlighted key opportunities for improving customer retention and revenue. Approximately 35% of customers were identified as one-time buyers, leading to the creation of dedicated one-time customer segments that improved segmentation coverage and reduced the proportion of customers classified as "Other" from approximately 11.5% to 1.3%.

The analysis showed that Champions are the most valuable customers and should be retained through loyalty-focused initiatives. Potential Loyalists were identified as the primary target for growth-focused campaigns due to their recent engagement and demonstrated repeat purchase behaviour. Additionally, At Risk customers represent a valuable segment that can potentially be recovered through targeted win-back campaigns.

The resulting segmentation framework can be used by the business to support targeted marketing campaigns, improve customer retention, and maximize customer lifetime value.